In [ ]:
import numpy as np

from miscope import load_family
from miscope.visualization.renderers.freq_group_weight_geometry import (
    render_weight_geometry_centroid_pca,
    render_weight_geometry_timeseries,
)

In [ ]:
PRIME, SEED, DATA_SEED, MATRIX = 113, 999, 999, "Wout"

family = load_family("modulo_addition_1layer")
variant = family.get_variant(prime=PRIME, seed=SEED, data_seed=DATA_SEED)

In [ ]:
fig = variant.view("weight_geometry.timeseries", matrix=MATRIX).figure()
fig.show()

In [ ]:
fig = variant.view("weight_geometry.group_snapshot", matrix=MATRIX).figure()
fig.show()

In [ ]:
fig = variant.view("weight_geometry.centroid_pca", matrix=MATRIX).figure()
fig.show()

In [ ]:
data = variant.artifacts.load_cross_epoch("neuron_group_pca")
projections = data["projections"]   # (n_epochs, d_mlp, 3)
group_freqs = data["group_freqs"]
neuron_group_idx = data["neuron_group_idx"]

final_proj = projections[-1]        # (d_mlp, 3) — final epoch


def fit_group_surface(proj_group):
    """
    Two-stage quadratic surface fit of PC3 against PC1, PC2.
    
    Stage 1 (linear):  PC3 = d*PC1 + e*PC2 + f
    Stage 2 (quadratic): adds a*PC1^2 + b*PC2^2 + c*PC1*PC2
    
    Returns r2_linear, r2_quadratic, a, b, and shape label.
    The incremental R2 (quadratic - linear) isolates actual curvature
    from tilt, so a flat-but-tilted plane scores low curvature R2.
    """
    pc1, pc2, pc3 = proj_group[:, 0], proj_group[:, 1], proj_group[:, 2]
    ss_tot = np.sum((pc3 - pc3.mean()) ** 2)
    if ss_tot < 1e-12:
        return dict(r2_linear=0, r2_quadratic=0, r2_curvature=0,
                    a=0, b=0, shape="flat")

    def r2(X, y):
        coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ coeffs
        return 1.0 - np.sum(resid**2) / ss_tot, coeffs

    X_lin  = np.column_stack([pc1, pc2, np.ones(len(pc1))])
    X_quad = np.column_stack([pc1**2, pc2**2, pc1*pc2, pc1, pc2, np.ones(len(pc1))])

    r2_lin,  _        = r2(X_lin,  pc3)
    r2_quad, coeffs_q = r2(X_quad, pc3)

    a, b = coeffs_q[0], coeffs_q[1]
    r2_curv = r2_quad - r2_lin  # variance explained by curvature alone

    if r2_curv < 0.05:
        shape = "flat/blob"
    elif np.sign(a) == np.sign(b):
        shape = "bowl"
    else:
        shape = "saddle"

    return dict(r2_linear=r2_lin, r2_quadratic=r2_quad, r2_curvature=r2_curv,
                a=a, b=b, shape=shape)


print(f"prime={PRIME}, seed={SEED}, data_seed={DATA_SEED}")
print(f"{'Freq':>6}  {'n':>4}  {'R²_lin':>7}  {'R²_quad':>8}  {'R²_curv':>8}  {'a':>8}  {'b':>8}  Shape")
print("-" * 70)
for g_idx, freq in enumerate(group_freqs):
    mask = neuron_group_idx == g_idx
    proj_g = final_proj[mask]
    result = fit_group_surface(proj_g)
    print(
        f"{freq:>6}  {mask.sum():>4}  "
        f"{result['r2_linear']:>7.3f}  {result['r2_quadratic']:>8.3f}  "
        f"{result['r2_curvature']:>8.3f}  "
        f"{result['a']:>8.3f}  {result['b']:>8.3f}  {result['shape']}"
    )


prime=59, seed=485, data_seed=999
  Freq     n   R²_lin   R²_quad   R²_curv         a         b  Shape
----------------------------------------------------------------------
     1    32   -0.000     0.084     0.084    -0.140    -0.303  bowl
     4   134   -0.000     0.019     0.019     0.004     0.111  flat/blob
     7    30    0.000     0.056     0.056     0.245    -0.175  saddle
    10    14   -0.000     0.602     0.602    -0.312     0.773  saddle
    14   185    0.000     0.020     0.020    -0.005    -0.027  flat/blob
    16    46    0.000     0.125     0.125     0.084     0.588  bowl
    27    69    0.000     0.014     0.014     0.004     0.079  flat/blob


In [ ]:

import pandas as pd

variants = family.list_variants()

def fit_group_surface(proj_group):
    pc1, pc2, pc3 = proj_group[:, 0], proj_group[:, 1], proj_group[:, 2]
    ss_tot = np.sum((pc3 - pc3.mean()) ** 2)
    if ss_tot < 1e-12:
        return dict(r2_linear=0, r2_quadratic=0, r2_curvature=0, a=0, b=0, shape="flat")

    def r2(X, y):
        coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
        return 1.0 - np.sum((y - X @ coeffs) ** 2) / ss_tot, coeffs

    X_lin  = np.column_stack([pc1, pc2, np.ones(len(pc1))])
    X_quad = np.column_stack([pc1**2, pc2**2, pc1*pc2, pc1, pc2, np.ones(len(pc1))])
    r2_lin, _         = r2(X_lin,  pc3)
    r2_quad, coeffs_q = r2(X_quad, pc3)
    a, b = coeffs_q[0], coeffs_q[1]
    r2_curv = r2_quad - r2_lin

    if r2_curv < 0.05:
        shape = "flat/blob"
    elif np.sign(a) == np.sign(b):
        shape = "bowl"
    else:
        shape = "saddle"

    return dict(r2_linear=r2_lin, r2_quadratic=r2_quad, r2_curvature=r2_curv,
                a=a, b=b, shape=shape)

*If
rows = []
for variant in variants:
    try:
        data = variant.artifacts.load_cross_epoch("neuron_group_pca")
    except FileNotFoundError:
        continue

    projections     = data["projections"]       # (n_epochs, d_mlp, 3)
    group_freqs     = data["group_freqs"]
    neuron_group_idx = data["neuron_group_idx"]
    final_proj      = projections[-1]

    p     = variant.params["prime"]
    seed  = variant.params["seed"]
    dseed = variant.params["data_seed"]

    for g_idx, freq in enumerate(group_freqs):
        mask   = neuron_group_idx == g_idx
        result = fit_group_surface(final_proj[mask])
        rows.append(dict(
            prime=p, seed=seed, data_seed=dseed,
            freq=int(freq), n=int(mask.sum()),
            **result,
        ))

df = pd.DataFrame(rows)
print(df.groupby("shape")["r2_curvature"].describe().round(3))
print()
print(df.groupby(["prime", "seed", "data_seed"])[["r2_curvature"]].mean().round(3).to_string())


           count   mean    std    min    25%    50%    75%    max
shape                                                            
bowl        32.0  0.643  0.234  0.070  0.625  0.736  0.803  0.885
flat/blob    3.0  0.018  0.003  0.014  0.017  0.019  0.020  0.020
saddle     113.0  0.745  0.159  0.056  0.667  0.779  0.860  0.949

                      r2_curvature
prime seed data_seed              
59    485  42                0.787
           598               0.592
           999               0.132
      999  598               0.775
           999               0.652
89    485  598               0.784
           999               0.742
      999  598               0.794
           999               0.582
97    42   598               0.820
      485  598               0.698
      999  598               0.494
101   485  42                0.814
           598               0.760
           999               0.718
      999  42                0.761
           598               0.819
    